## Calibration

In [1]:
import cv2
import math
import numpy as np
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\user\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [9]:
CALIBRATION_IMAGE_PATH = "calibration.jpg"  # Your image file
OUTPUT_IMAGE_PATH = "calibration_verified.jpg" # Where to save the result
REAL_LENGTH_METERS = 4.227  # Known length of your reference car
MODEL_PATH = "yolov8n-seg.pt" 

In [17]:
def get_robust_geometry(mask_points):
    """
    Calculates the oriented bounding box of the mask.
    Returns:
      - length_px: The longest dimension of the box (assuming it's length)
      - box_points: The 4 corner points of the rotated box for drawing
    """
    # 1. Convert mask to numpy float32 format required by minAreaRect
    contour = np.array(mask_points, dtype=np.float32)

    # 2. Compute the Minimum Area Rectangle (Rotated Box)
    # Returns ((center_x, center_y), (width, height), angle)
    rect = cv2.minAreaRect(contour)
    (width, height) = rect[1]

    # 3. Determine Length (Longest side)
    length_px = max(width, height)

    # 4. Get the 4 corner points of the box (for visualization)
    box_points = cv2.boxPoints(rect)
    box_points = np.intp(box_points) # Convert to integer for drawing

    return length_px, box_points

def run_calibration_with_visuals():
    print(f"Loading {CALIBRATION_IMAGE_PATH}...")
    img = cv2.imread(CALIBRATION_IMAGE_PATH)
    if img is None:
        print("Error: Could not read image.")
        return

    model = YOLO(MODEL_PATH)

    # --- INFERENCE ---
    results = model(img)[0]

    # Find the best vehicle mask
    best_mask_data = None
    max_conf = 0

    if results.masks is None:
        print("Error: No masks found. Is the car clearly visible?")
        return

    for i, mask_pts in enumerate(results.masks.xy):
        conf = results.boxes.conf[i]
        cls = int(results.boxes.cls[i])
        
        # Check for Car(2), Bus(5), Truck(7)
        if cls in [2, 5, 7]: 
            if conf > max_conf:
                max_conf = conf
                best_mask_data = mask_pts

    if best_mask_data is None:
        print("Error: No vehicle detected.")
        return

    # --- CALCULATION (ROBUST) ---
    pixel_length, box_points = get_robust_geometry(best_mask_data)

    if pixel_length == 0:
        print("Error: Calculated length is 0.")
        return

    ppm = pixel_length / REAL_LENGTH_METERS

    # --- VISUALIZATION ---
    
    # 1. Draw the Segmentation Mask (Green)
    # Reshape for polylines: (N, 1, 2)
    mask_poly = np.array(best_mask_data, np.int32).reshape((-1, 1, 2))
    cv2.polylines(img, [mask_poly], True, (0, 255, 0), 2)

    # 2. Draw the Rotated Bounding Box (Red)
    # This shows exactly what the algorithm measured to get the length
    cv2.drawContours(img, [box_points], 0, (0, 0, 255), 2)

    # 3. Add Text Info
    info_text = f"Length: {pixel_length:.1f}px | PPM: {ppm:.2f}"
    
    # Text background for readability
    (w, h), _ = cv2.getTextSize(info_text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
    cv2.rectangle(img, (10, 10), (10 + w, 10 + h + 20), (0, 0, 0), -1)
    
    cv2.putText(img, info_text, (10, 35), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    # Save Output
    cv2.imwrite(OUTPUT_IMAGE_PATH, img)
    print("------------------------------------------------")
    print(f"Calibration Successful using Rotated Rect!")
    print(f"PPM: {ppm:.4f}")
    print(f"Visual saved to: {OUTPUT_IMAGE_PATH}")
    print("------------------------------------------------")

In [18]:
run_calibration_with_visuals()

Loading calibration.jpg...

0: 384x640 5 cars, 20.6ms
Speed: 15.3ms preprocess, 20.6ms inference, 10.9ms postprocess per image at shape (1, 3, 384, 640)
------------------------------------------------
Calibration Successful using Rotated Rect!
PPM: 60.3265
Visual saved to: calibration_verified.jpg
------------------------------------------------


In [2]:
# Method 1: Auto-download on first use
from ultralytics import YOLO

# This will automatically download yolo11n-seg.pt if not present
model = YOLO('yolo11n-seg.pt')